In [1]:

import numpy as np


# -----------------------------------------
# 1. Gaussian Elimination - Partial Pivoting
# -----------------------------------------

def partial_pivoting(A, B):
    A = A.astype(float).copy()
    B = B.astype(float).copy()

    n = len(B)

    for k in range(n - 1):

        # Find the largest element in the current column
        pivot = k + np.argmax(np.abs(A[k:, k]))

        # Exchange rows if necessary
        if pivot != k:
            A[[k, pivot]] = A[[pivot, k]]
            B[[k, pivot]] = B[[pivot, k]]

        if abs(A[k, k]) < 1e-12:
            raise ValueError("Singular or nearly singular matrix")

        # Elimination
        for i in range(k + 1, n):
            factor = A[i, k] / A[k, k]

            A[i, k:] -= factor * A[k, k:]
            B[i] -= factor * B[k]

    if abs(A[n - 1, n - 1]) < 1e-12:
        raise ValueError("Singular or nearly singular matrix")

    # Backward substitution
    X = np.zeros(n)

    for i in range(n - 1, -1, -1):
        X[i] = (
            B[i] - np.dot(A[i, i + 1:], X[i + 1:])
        ) / A[i, i]

    return X


# -----------------------------------------
# 2. Gaussian Elimination - Complete Pivoting
# -----------------------------------------

def complete_pivoting(A, B):
    A = A.astype(float).copy()
    B = B.astype(float).copy()

    n = len(B)

    # Track the original order of variables
    perm = np.arange(n)

    for k in range(n - 1):

        # Find the largest element in the remaining submatrix
        submatrix = np.abs(A[k:, k:])
        row, col = np.unravel_index(
            np.argmax(submatrix), submatrix.shape
        )

        pivot_row = k + row
        pivot_col = k + col

        # Exchange rows
        if pivot_row != k:
            A[[k, pivot_row]] = A[[pivot_row, k]]
            B[[k, pivot_row]] = B[[pivot_row, k]]

        # Exchange columns
        if pivot_col != k:
            A[:, [k, pivot_col]] = A[:, [pivot_col, k]]
            perm[[k, pivot_col]] = perm[[pivot_col, k]]

        if abs(A[k, k]) < 1e-12:
            raise ValueError("Singular or nearly singular matrix")

        # Elimination
        for i in range(k + 1, n):
            factor = A[i, k] / A[k, k]

            A[i, k:] -= factor * A[k, k:]
            B[i] -= factor * B[k]

    if abs(A[n - 1, n - 1]) < 1e-12:
        raise ValueError("Singular or nearly singular matrix")

    # Backward substitution
    Y = np.zeros(n)

    for i in range(n - 1, -1, -1):
        Y[i] = (
            B[i] - np.dot(A[i, i + 1:], Y[i + 1:])
        ) / A[i, i]

    # Restore original variable order
    X = np.zeros(n)
    X[perm] = Y

    return X


# -----------------------------------------
# 3. Main Program
# -----------------------------------------

A = np.array([
    [2, 1, -1],
    [3, 2, 1],
    [1, -1, 2]
], dtype=float)

B = np.array([8, 11, -3], dtype=float)

# Keep original data for residual calculation
A_original = A.copy()
B_original = B.copy()

# Apply both methods
X_partial = partial_pivoting(A, B)
X_complete = complete_pivoting(A, B)

# Calculate residuals
residual_partial = np.linalg.norm(
    A_original @ X_partial - B_original
)

residual_complete = np.linalg.norm(
    A_original @ X_complete - B_original
)

# Display results
print("===== PARTIAL PIVOTING =====")
print("Solution:", X_partial)
print("Residual:", residual_partial)

print("\n===== COMPLETE PIVOTING =====")
print("Solution:", X_complete)
print("Residual:", residual_complete)

print("\n===== COMPARISON =====")
print("Both methods produce the same solution:",
      np.allclose(X_partial, X_complete))


===== PARTIAL PIVOTING =====
Solution: [ 2.  3. -1.]
Residual: 4.440892098500626e-16

===== COMPLETE PIVOTING =====
Solution: [ 2.  3. -1.]
Residual: 4.440892098500626e-16

===== COMPARISON =====
Both methods produce the same solution: True
